In [1]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

2026-03-11 17:59:33.099764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773251973.282510      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773251973.328656      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773251973.725320      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773251973.725368      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773251973.725371      55 computation_placer.cc:177] computation placer alr

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

base_path = "/kaggle/input/datasets/iarunava/cell-images-for-detecting-malaria/cell_images/cell_images"

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=180,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest"
)
train_generator = train_datagen.flow_from_directory(
    base_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical', 
    subset='training'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)
val_generator = val_datagen.flow_from_directory(
    base_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 22048 images belonging to 2 classes.
Found 5510 images belonging to 2 classes.


In [4]:
print(train_generator.class_indices)
print(train_generator.num_classes)

{'Parasitized': 0, 'Uninfected': 1}
2


In [5]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint,ReduceLROnPlateau, CSVLogger

callbacks = [
    ModelCheckpoint(
        filepath="resnet50_malaria.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_accuracy",
        patience=7,              # give ReduceLR time to work first
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,              # halve LR when plateauing
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    CSVLogger("malaria_log.csv") # saves epoch-by-epoch to CSV
                                  # useful if Kaggle session dies
]

In [6]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

In [7]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(128,128,3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
outputs = Dense(train_generator.num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)

I0000 00:00:1773252106.168553      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [ ]:
model.summary()

In [8]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
history1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[callbacks]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20


I0000 00:00:1773252154.538039     143 service.cc:152] XLA service 0x79dc18003ec0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773252154.538082     143 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773252156.344011     143 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/689 ━━━━━━━━━━━━━━━━━━━━ 2:16:16 12s/step - accuracy: 0.7812 - loss: 1.9363

I0000 00:00:1773252159.862721     143 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step - accuracy: 0.7999 - loss: 0.5589
Epoch 1: val_accuracy improved from -inf to 0.86334, saving model to resnet50_malaria.keras
689/689 ━━━━━━━━━━━━━━━━━━━━ 243s 336ms/step - accuracy: 0.8000 - loss: 0.5586 - val_accuracy: 0.8633 - val_loss: 0.3111 - learning_rate: 1.0000e-04
Epoch 2/20
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - accuracy: 0.8980 - loss: 0.2558
Epoch 2: val_accuracy improved from 0.86334 to 0.87060, saving model to resnet50_malaria.keras
689/689 ━━━━━━━━━━━━━━━━━━━━ 146s 212ms/step - accuracy: 0.8980 - loss: 0.2558 - val_accuracy: 0.8706 - val_loss: 0.2992 - learning_rate: 1.0000e-04
Epoch 3/20
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.9098 - loss: 0.2320
Epoch 3: val_accuracy improved from 0.87060 to 0.88312, saving model to resnet50_malaria.keras
689/689 ━━━━━━━━━━━━━━━━━━━━ 147s 214ms/step - accuracy: 0.9098 - loss: 0.2320 - val_accuracy: 0.8831 - val_loss: 0.2722 - learning_rate: 1.0000e-04
Epoch 4/20
689/689

In [11]:
model.save("resnet50_malaria.keras")